# Silver Layer Edge Case Analysis

This notebook tests how the Silver Layer handles common real-world data anomalies: price fluctuations, daily metadata noise, and attribute drift.

In [38]:
import os
import duckdb
from dotenv import load_dotenv
load_dotenv("../.env")

database = os.getenv('MD_DATABASE', 'my_db')
token = os.getenv('MOTHERDUCK_TOKEN')
con = duckdb.connect(f"md:{database}?token={token}")

## Case 1: The 'Price Yo-Yo'
When a seller changes the price up and down multiple times, we should see a clean SCD Type 2 chain without losing any intermediate states.

In [39]:
# Example: olx-1addYn (Identified in audit as having 8 price changes)
yoyo_query = """
SELECT 
    valid_from, 
    valid_to, 
    price_total, 
    is_current
FROM silver.listing_versions 
WHERE source_listing_id = 'olx-1addYn'
ORDER BY valid_from ASC
"""
display(con.execute(yoyo_query).df())

,valid_from,valid_to,price_total,is_current
0,2026-04-15 07:37:44.475955,2026-04-17 07:40:22.714804,1800.0,False
1,2026-04-17 07:40:22.714804,2026-04-18 07:21:45.531481,1700.0,False
2,2026-04-18 07:21:45.531481,2026-04-22 07:39:12.142455,1800.0,False
3,2026-04-22 07:39:12.142455,2026-04-23 07:44:07.474238,1600.0,False
4,2026-04-23 07:44:07.474238,2026-04-25 07:26:56.712288,1800.0,False
5,2026-04-25 07:26:56.712288,2026-04-30 08:16:01.812162,1600.0,False
6,2026-04-30 08:16:01.812162,2026-05-12 08:26:24.633689,1300.0,False
7,2026-05-12 08:26:24.633689,2026-05-13 08:38:50.119496,1400.0,False
8,2026-05-13 08:38:50.119496,NaT,1300.0,True


## Case 2: Daily Metadata Noise (Snapshot Deduplication)
OLX daily snapshots often contain NO functional changes. Silver should collapse these into a single version.

In [40]:
noise_query = """
SELECT 
    source_listing_id, 
    count(*) as version_count,
    min(valid_from) as first_seen,
    max(valid_from) as latest_version_start
FROM silver.listing_versions 
GROUP BY 1 
HAVING version_count = 1
LIMIT 5
"""
print("Listings that have been snapped daily but remained functionally identical (1 Version Only):")
display(con.execute(noise_query).df())

Listings that have been snapped daily but remained functionally identical (1 Version Only):


,source_listing_id,version_count,first_seen,latest_version_start
0,olx-15FAAM,1,2026-05-21 09:05:50.980150,2026-05-21 09:05:50.980150
1,olx-17a0Ti,1,2026-04-16 07:42:43.417848,2026-04-16 07:42:43.417848
2,olx-19BfGk,1,2026-04-06 07:38:34.901408,2026-04-06 07:38:34.901408
3,olx-19QMoS,1,2026-04-09 07:26:11.502399,2026-04-09 07:26:11.502399
4,olx-19XnPi,1,2026-04-20 08:01:56.277210,2026-04-20 08:01:56.277210


## Case 3: Identity Collision Protection
Does the model correctly separate a listing if the same ID were to appear in both Rent and Sale?

In [41]:
collision_query = """
SELECT 
    source_listing_id, 
    count(distinct mode) as mode_count
FROM silver.listing_identity
GROUP BY 1
HAVING mode_count > 1
"""
results = con.execute(collision_query).df()
if results.empty:
    print("No current ID collisions found across modes, but the PK constraint (source, source_listing_id, mode) protects against this.")
else:
    print("Active ID collisions across Rent/Sale managed correctly:")
    display(results)

No current ID collisions found across modes, but the PK constraint (source, source_listing_id, mode) protects against this.


In [42]:
con.close()